# Libraries

In [34]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

from scipy.signal import welch


import xgboost
from sklearn.neighbors import KNeighborsClassifier



## Basic inspection/loading

In [35]:
dataset_path = Path('DEED')
eeg_dataset = []
for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  # remove ".mat"
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    label = int(label_part[1:])  # convert E3 → 3
    
    eeg_dataset.append((eeg, label))

print(f"Loaded {len(eeg_dataset)} trials.")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

Loaded 533 trials.
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]


In [36]:
# * Verification of labels
for file in dataset_path.iterdir():
    fname = file.stem
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    label = int(label_part[1:])
    print(f"Filename: {fname} → Extracted label: {label}")

Filename: G_S0321_M1_E2_R1_N2_raw_ref → Extracted label: 2
Filename: G_S0213_M3_E2_R7_N2_raw_ref → Extracted label: 2
Filename: G_S0393_M2_E3_R5_REM_raw_ref → Extracted label: 3
Filename: G_S0243_M3_E2_R2_N2_raw_ref → Extracted label: 2
Filename: G_S0311_M3_E5_R2_N2_raw_ref → Extracted label: 5
Filename: G_S0031_M1_E3_R4_nan_raw_ref → Extracted label: 3
Filename: G_S0043_M2_E2_R5_N2_raw_ref → Extracted label: 2
Filename: G_S0072_M1_E0_R11_N1_raw_ref → Extracted label: 0
Filename: G_S0242_M1_E3_R3_W_raw_ref → Extracted label: 3
Filename: G_S0342_M2_E3_R3_N2_raw_ref → Extracted label: 3
Filename: G_S0033_M2_E4_R3_W_raw_ref → Extracted label: 4
Filename: G_S0023_M1_E2_R1_N1_raw_ref → Extracted label: 2
Filename: G_S0302_M3_E2_R3_N1_raw_ref → Extracted label: 2
Filename: G_S0152_M2_E4_R3_N1_raw_ref → Extracted label: 4
Filename: G_S0373_M3_E4_R1_N1_raw_ref → Extracted label: 4
Filename: G_S0043_M2_E3_R3_N2_raw_ref → Extracted label: 3
Filename: G_S0283_M1_E2_R2_N3_raw_ref → Extracted label

### Segmentation: 2s window based on Moctezuma

In [37]:
def segmentation(eeg_dataset, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []

    for eeg_array, label in eeg_dataset:
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            start += window_size  

    return X, y

### Segment into 2, 10, 20s windows

In [38]:
secseg2, secseg2_labels = segmentation(eeg_dataset, 2, 200)
secseg10, secseg10_labels = segmentation(eeg_dataset, 10, 200)
secseg20, secseg20_labels = segmentation(eeg_dataset, 20, 200)
print(f"2s windows: {len(secseg2)}, 10s windows: {len(secseg10)}, 20s windows: {len(secseg20)}")

2s windows: 76657, 10s windows: 15245, 20s windows: 7490


# Feature Extraction
- 1. PSD (Power Spectral Density)

In [39]:
freq_bands = {'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)}


In [40]:
def extract_psd_features(segmented_windows, labels, fs=200):

    X = []
    for window in segmented_windows:
        # window: shape (n_channels, n_samples)
        window_features = []
        for ch_signal in window:
            # Compute PSD using Welch
            f, Pxx = welch(ch_signal, fs=fs, nperseg=len(ch_signal), noverlap=0)

            # Compute band powers
            for band in freq_bands.values():
                idx = np.logical_and(f >= band[0], f <= band[1])
                band_power = np.mean(Pxx[idx])
                window_features.append(band_power)

        X.append(window_features)

    X = np.array(X)
    y = np.array(labels)

    return X, y

In [41]:
X2, y2 = extract_psd_features(secseg2, secseg2_labels)
print(X2.shape)  # (n_windows, n_channels * n_bands)
print(y2.shape)  # (n_windows,)

(76657, 30)
(76657,)


In [42]:
X10, y10 = extract_psd_features(secseg10, secseg10_labels)

print(X10.shape)  
print(y10.shape)  

(15245, 30)
(15245,)


In [44]:
X20, y20 = extract_psd_features(secseg20, secseg20_labels)

print(X20.shape)
print(y20.shape)  # (n_windows,)

(7490, 30)
(7490,)
